# Implementing basic ViT (vision transformer) from scratch

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#data transformers
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from einops import repeat
from einops.layers.torch import Rearrange


In [11]:
# using mps if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [12]:
#input image dataset and transformers
batch_size = 256

#load cifar10 dataset with data augmentations
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)

test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

Files already downloaded and verified
Files already downloaded and verified


### Initilize the multihead attention block

In [ ]:
#attention mechanism
class MultiHeadAttention(nn.Module):
    
    def __init__(self, embed_size, heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        
        self.heads = heads
        self.attention = nn.MultiheadAttention(embed_size, heads, dropout=dropout, batch_first=True)

        self.q = nn.Linear(embed_size, embed_size)
        self.k = nn.Linear(embed_size, embed_size)
        self.v = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        
        q = q.transpose(0, 1)  # (seq_len, batch_size, embed_size)
        k = k.transpose(0, 1)  # (seq_len, batch_size, embed_size)
        v = v.transpose(0, 1)  # (seq_len, batch_size, embed_size)

        out, _ = self.attention(q, k, v)
        out = out.transpose(0, 1)  # (batch_size, seq_len, embed_size)
        
        return out

### Pre-normalization 

In [14]:
class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

### Feed forward class

In [15]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super(FeedForward, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

# Residual Connections

In [16]:
class ResConnect(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

# MAIN ViT Implementation

In [17]:
class ViT(nn.Module): 
    
    def __init__(self, c=3, image_size=32, patch_size=8, num_classes=10, dim=128, depth=6, heads=8, mlp_dim=256, dropout=0.1):
        super().__init__()

        # Model architecture will be defined here
        # including patch embedding, positional encoding, transformer encoder layers, and classification head
        self.c = c
        self.image_size = image_size            # Input image size
        self.patch_size = patch_size      # Size of each patch
        self.num_classes = num_classes  # Number of output classes
        self.dim = dim                  # Embedding dimension
        self.depth = depth              # Number of transformer blocks
        self.heads = heads              # Number of attention heads
        self.mlp_dim = mlp_dim          # Dimension of the feedforward network
        self.dropout = dropout        # Dropout rate
        
        #initilaized model 
        print(f"Initializing ViT model with image_size={image_size}, patch_size={patch_size}, num_classes={num_classes}, dim={dim}, depth={depth}, heads={heads}, mlp_dim={mlp_dim}, dropout={dropout}")
        
        # Patch embedding
        self.patch_embed =  nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch_size, p2=patch_size),
            nn.Linear(patch_size * patch_size * c, dim)
        )

        print(f"Patch embedding layer and shape: {self.patch_embed}, output dim: {dim}")

        # Positional encoding and class token
        num_patches = (image_size // patch_size) ** 2
        
        # Positional embedding
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        
        # Class token
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.dropout = nn.Dropout(dropout)
        
        
        # Transformer Encoder layers
        self.transformer = nn.ModuleList([])
        for _ in range(depth):
            self.transformer.append(nn.ModuleList([
                ResConnect(PreNorm(dim, MultiHeadAttention(embed_size=dim, heads=heads, dropout=dropout))),
                ResConnect(PreNorm(dim, FeedForward(dim=dim, hidden_dim=mlp_dim, dropout=dropout)))
            ]))
            
        #classification head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )
    
    def forward(self, x):
        
        # Prepare patches
        N, C, H, W = x.shape
        x = self.patch_embed(x)  # (N, dim, H/patch_size, W/patch_size)
            
        #add cls token to inputs 
        cls_tokens = repeat(self.cls_token, '1 1 d -> n 1 d', n=N)  # (N, 1, dim)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :x.size(1), :]
        
        #transformer layers
        for i in range(self.depth):
            attn, ff = self.transformer[i]
            x = attn(x)
            x = ff(x)
            
        return self.head(x[:, 0])

model = ViT()
print(model)

Initializing ViT model with image_size=32, patch_size=8, num_classes=10, dim=128, depth=6, heads=8, mlp_dim=256, dropout=0.1
Patch embedding layer and shape: Sequential(
  (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=8, p2=8)
  (1): Linear(in_features=192, out_features=128, bias=True)
), output dim: 128
ViT(
  (patch_embed): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=8, p2=8)
    (1): Linear(in_features=192, out_features=128, bias=True)
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): ModuleList(
    (0-5): 6 x ModuleList(
      (0): ResConnect(
        (fn): PreNorm(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (fn): MultiHeadAttention(
            (attention): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
            )
            (q): Linear(in_features=128, out_features=128, bias=True)
            (k): Line

## Training

In [18]:
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

idx = 0

## Training
epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        batch_acc = (preds == labels).float().mean().item()
        print(f"Epoch [{epoch+1}/{epochs}]  |  batch: {idx+1}/{len(train_loader)}  |  Loss: {loss.item():.4f}  |  batch accuracy: {batch_acc:.4f}", flush=True, end='\r')

    avg_loss = total_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print("--"*10, f"Epoch [{epoch+1}/{epochs}]  |  Loss: {avg_loss:.4f} |  accuracy: {epoch_acc:.2f}%", '--'*10)

    # Evaluation on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = 100 * correct / total
    print(f"Test Accuracy: {test_acc:.2f}%")
            



-------------------- Epoch [1/20]  |  Loss: 2.3187 |  accuracy: 9.92% --------------------
Test Accuracy: 10.00%
-------------------- Epoch [2/20]  |  Loss: 2.3092 |  accuracy: 9.68% --------------------
Test Accuracy: 10.00%
-------------------- Epoch [3/20]  |  Loss: 2.3083 |  accuracy: 9.83% --------------------
Test Accuracy: 10.00%
-------------------- Epoch [4/20]  |  Loss: 2.3083 |  accuracy: 10.00% --------------------
Test Accuracy: 10.00%


KeyboardInterrupt: 